# Lista 6 - Grupo 06

Monte Carlo CPM/PERT: risco de prazo, risco de custo, probabilidade de caminho crítico e correlação prazo-custo, para diferentes valores de coeficiente de correlação ρ entre as atividades.

In [ ]:
install.packages('triangle')
install.packages('igraph')
install.packages('MASS')


## Modelando o problema

Dados da rede de precedências, parâmetros das distribuições triangulares de duração e custo, e construção do grafo de atividades.

In [ ]:
library(igraph)
library(triangle)
library(MASS)

# Atividades: A B C D E F G H I J
# Nós do grafo: 1=START, 2=A, 3=B, 4=C, 5=D, 6=E, 7=F, 8=G, 9=H, 10=I, 11=J... 
# (conferir mapeamento abaixo via Pre/Suc)

# Estruturas de dados globais
n  <- 11      # número de nós do grafo (START + 10 atividades), END tratado como nó 11... ver abaixo
Ns <- 3000    # número de cenários (réplicas) do Monte Carlo

# DURAÇÃO (DMin, DMp, DMax) - triangular - linhas na ordem A..J
parDur <- matrix(data = c(
   2,  4, 18,   # A - Obter materiais
   5,  9, 19,   # B - Obter mão de obra
   4, 10, 28,   # C - Escavar
   8, 13, 36,   # D - Colocar fundação
  44, 60,100,   # E - Construir estrutura
  30, 40, 74,   # F - Instalação Hidráulica
   9, 20, 43,   # G - Instalação Elétrica
  24, 30, 48,   # H - Acabamento interior
  28, 29, 96,   # I - Acabamento Exterior
  10, 10, 12    # J - Limpeza Local
), ncol = 3, byrow = TRUE)

# CUSTO (CMin, CMp, CMax) - triangular - linhas na ordem A..J
parCus <- matrix(data = c(
    300,   450,   600,   # A
    480,   600,   720,   # B
   3750,  4500,  5250,   # C
   8400,  9600, 10800,   # D
 300000,312000,322500,   # E
  37650, 39600, 41400,   # F
  10500, 11550, 12600,   # G
  36000, 38400, 40800,   # H
  48750, 52500, 56250,   # I
    360,   450,   540    # J
), ncol = 3, byrow = TRUE)

# Rede de precedências
# Nó 0 = START (fictício), Nó 10 = END (fictício) - antes do +1
# A=1, B=2, C=3, D=4, E=5, F=6, G=7, H=8, I=9, J=10
#
# Predecessores conforme tabela:
#   A: nenhum  -> START -> A
#   B: nenhum  -> START -> B
#   C: A       -> A -> C
#   D: C       -> C -> D
#   E: B, D    -> B -> E, D -> E
#   F: E       -> E -> F
#   G: E       -> E -> G
#   H: F, G    -> F -> H, G -> H
#   I: F       -> F -> I
#   J: H, I    -> H -> J, I -> J
#   J -> END

# Arestas: (origem, destino) — usando nó 0=START, 10=END
elos <- c(
  0,1,   # START -> A
  0,2,   # START -> B
  1,3,   # A -> C
  3,4,   # C -> D
  2,5,   # B -> E
  4,5,   # D -> E
  5,6,   # E -> F
  5,7,   # E -> G
  6,8,   # F -> H
  7,8,   # G -> H
  6,9,   # F -> I
  8,10,  # H -> J
  9,10   # I -> J
)

# Listas de sucessores
Suc <- list(
  c(1,2),    # 0=START -> A, B
  3,         # 1=A -> C
  5,         # 2=B -> E
  4,         # 3=C -> D
  5,         # 4=D -> E
  c(6,7),    # 5=E -> F, G
  c(8,9),    # 6=F -> H, I
  8,         # 7=G -> H
  10,        # 8=H -> J
  10,        # 9=I -> J
  0,         # 10=J -> END
  0          # END
)

# Listas de predecessores
Pre <- list(
  0,         # 0=START (sem predecessores)
  0,         # 1=A <- START
  0,         # 2=B <- START
  1,         # 3=C <- A
  3,         # 4=D <- C
  c(2,4),    # 5=E <- B, D
  5,         # 6=F <- E
  5,         # 7=G <- E
  c(6,7),    # 8=H <- F, G
  6,         # 9=I <- F
  c(8,9),    # 10=J <- H, I
  10         # 11=END <- J
)

# Construção do grafo e caminhos simples
# +1 porque igraph usa índices a partir de 1 (nó 0=START -> índice 1, nó 10=END -> índice 11)
g <- make_graph(elos + 1)
plot(g, vertex.color = 'white',
     vertex.label = c('Inicio','A','B','C','D','E','F','G','H','I','J','Fim'))

sp <- all_simple_paths(g, from = 1, to = 12)  # 1=START, 12=END
l  <- length(sp)
cat("Número de caminhos simples:", l, "\n")


**Observação sobre a numeração dos nós:** o grafo tem 12 nós no total (`vertex.label` com 12 elementos: Início, A..J, Fim). Portanto `n <- 12` (não 11), com nós 1=Início, 2..11=A..J, 12=Fim. As colunas das matrizes `dur` e `cus` (Ns x n) seguem essa mesma numeração: coluna 1 e coluna 12 ficam sempre em zero (START/END não têm duração/custo), e as colunas 2..11 correspondem às atividades A..J (`parDur[i-1,]`/`parCus[i-1,]` para coluna `i`).

In [ ]:
n <- 12  # corrige: 12 nós (Início + 10 atividades + Fim)


## Funções auxiliares: CPM determinístico

In [ ]:
funcCpm <- function(n=9,
                    d=c(0,2,6,4,3,5,4,2,0),
                    Suc=list(c(2,3,4),c(5,7),8,6,8,8,9,9,0),
                    Pre=list(0,1,1,1,2,4,2,c(3,5,6),c(7,8)))
{
  est <- rep(0, n)
  eft <- rep(0, n)
  lst <- rep(0, n)
  lft <- rep(0, n)

  for(i in 1:n){
    if(Pre[[i]][1] != 0){
      est[i] <- max(eft[Pre[[i]]])
    }
    eft[i] <- est[i] + d[i]
  }

  lft[n] <- eft[n]
  lst[n] <- lft[n] - d[n]

  for(i in (n-1):1){
    if(Suc[[i]][1] != 0){
      lft[i] <- min(lst[Suc[[i]]])
    } else {
      lft[i] <- eft[n]
    }
    lst[i] <- lft[i] - d[i]
  }

  slack <- lst - est

  list(est=est, lst=lst, slack=slack, duracao=eft[n])
}


### Cronograma determinístico (cenário mais provável)

Utiliza as durações mais prováveis (DMp) de cada atividade como sanity check do CPM antes da simulação.

In [ ]:
d <- c(0, parDur[,2], 0)   # d[1]=Inicio, d[2..11]=A..J (DMp), d[12]=Fim
r <- funcCpm(n, d, Suc, Pre)

cronograma <- data.frame(
  atividade = LETTERS[1:10],
  inicio    = r$est[2:11],
  duracao   = parDur[,2],   # duração mais provável
  folga     = r$slack[2:11]
)

print(cronograma)
cat("Duração total do projeto (cenário mais provável):", r$duracao, "\n")


## Simulação de Monte Carlo com correlação

Para cada valor de ρ, geramos amostras correlacionadas de duração e custo das 10 atividades usando o modelo:

1. `Z ~ MVN(0, A)` onde `A` é a matriz de correlação 10x10 com ρ constante fora da diagonal;
2. `U = Φ(Z)` (transformação para uniformes via `pnorm`);
3. Duração e custo de cada atividade obtidos por `qtriangle(U, min, max, mp)`.

O mesmo `U` é usado para duração e custo, de forma que a duração e o custo de uma mesma atividade também fiquem correlacionados (opção B), o que produz uma dispersão prazo×custo mais informativa.

In [ ]:
gerarAmostras <- function(rho, Ns, n, parDur, parCus) {
  # Matriz de correlação 10x10 (atividades A..J), correlação rho fora da diagonal
  Acorr <- matrix(rho, nrow = 10, ncol = 10)
  diag(Acorr) <- 1

  # Amostras normais reduzidas correlacionadas
  Z <- mvrnorm(Ns, mu = rep(0, 10), Sigma = Acorr)

  # Transforma para uniformes (0,1)
  U <- pnorm(Z)

  # Durações correlacionadas via inversa da triangular
  durAt <- matrix(0, nrow = Ns, ncol = 10)
  for (i in 1:10) {
    durAt[, i] <- ceiling(qtriangle(U[, i], parDur[i,1], parDur[i,3], parDur[i,2]))
  }

  # Custos correlacionados via inversa da triangular (mesmo U)
  cusAt <- matrix(0, nrow = Ns, ncol = 10)
  for (i in 1:10) {
    cusAt[, i] <- qtriangle(U[, i], parCus[i,1], parCus[i,3], parCus[i,2])
  }

  # Matrizes (Ns x n): coluna 1 = Início, colunas 2..11 = A..J, coluna 12 = Fim
  dur <- cbind(0, durAt, 0)
  cus <- cbind(0, cusAt, 0)

  list(dur = dur, cus = cus)
}


### Função de simulação completa para um dado ρ

Encapsula: geração de amostras, cálculo do caminho crítico por cenário (prazo e custo), análise de risco de prazo/custo, probabilidade de cada atividade estar no caminho crítico, e dispersão prazo x custo.

In [ ]:
simularProjeto <- function(rho, Ns, n, l, sp, parDur, parCus) {

  amostras <- gerarAmostras(rho, Ns, n, parDur, parCus)
  dur <- amostras$dur
  cus <- amostras$cus

  # Para cada cenário, identifica o caminho crítico (maior duração) e sua duração
  durCC <- numeric(Ns)
  icc   <- integer(Ns)
  dc    <- numeric(l)

  for (i in 1:Ns) {
    d <- dur[i, ]
    for (j in 1:l) {
      dc[j] <- sum(d[sp[[j]]])
    }
    icc[i]   <- which.max(dc)
    durCC[i] <- dc[icc[i]]
  }

  # Custo do caminho crítico (mesmo caminho identificado pelo prazo)
  cusCC <- numeric(Ns)
  for (i in 1:Ns) {
    cusCC[i] <- sum(cus[i, sp[[icc[i]]]])
  }

  # Probabilidade de cada atividade estar no caminho crítico
  nomesAtiv <- LETTERS[1:10]
  contCC <- rep(0, 10)
  for (k in seq_len(Ns)) {
    caminho <- as.integer(sp[[icc[k]]])
    caminho <- caminho[caminho != 1]   # remove nó Início
    caminho <- caminho[caminho != n]   # remove nó Fim
    caminho <- caminho - 1             # nó 2=A -> idx 1, ..., nó 11=J -> idx 10
    contCC[caminho] <- contCC[caminho] + 1
  }
  probCC <- contCC / Ns

  list(
    durCC = durCC,
    cusCC = cusCC,
    icc = icc,
    probCC = probCC,
    nomesAtiv = nomesAtiv
  )
}


## Execução para os 5 valores de ρ

Executa a simulação completa para ρ ∈ {0.00, 0.25, 0.50, 0.75, 0.95}, armazenando os resultados de cada rodada para comparação posterior.

In [ ]:
rhos <- c(0.00, 0.25, 0.50, 0.75, 0.95)
resultados <- list()

for (rho in rhos) {

  set.seed(42)  # mesma semente para todos os rho -> comparação mais justa
  res <- simularProjeto(rho, Ns, n, l, sp, parDur, parCus)
  resultados[[as.character(rho)]] <- res

  cat("\n\n========================================\n")
  cat("           rho =", rho, "\n")
  cat("========================================\n")

  # --- Risco de prazo ---
  cat("\n--- Risco de Prazo ---\n")
  print(summary(res$durCC))
  hist(res$durCC,
       main = paste("Distribuição da Duração da Obra (rho =", rho, ")"),
       xlab = "Duração (dias)", col = "lightgreen", border = "white")
  plot(ecdf(res$durCC),
       main = paste("ECDF - Duração da Obra (rho =", rho, ")"),
       xlab = "Duração (dias)", ylab = "P(Duração <= x)")

  # --- Risco de custo ---
  cat("\n--- Risco de Custo ---\n")
  print(summary(res$cusCC))
  hist(res$cusCC,
       main = paste("Distribuição do Custo da Obra (rho =", rho, ")"),
       xlab = "Custo (R$)", col = "steelblue", border = "white")
  plot(ecdf(res$cusCC),
       main = paste("ECDF - Custo da Obra (rho =", rho, ")"),
       xlab = "Custo (R$)", ylab = "P(Custo <= x)")

  # --- Probabilidade de caminho crítico ---
  cat("\n--- Probabilidade de cada atividade estar no caminho crítico ---\n")
  resultadoCC <- data.frame(
    Atividade = res$nomesAtiv,
    Probabilidade = round(res$probCC, 4),
    Percentual = round(res$probCC * 100, 2)
  )
  print(resultadoCC)

  barplot(
    res$probCC * 100,
    names.arg = res$nomesAtiv,
    col = "steelblue",
    border = "black",
    main = paste("P(Atividade no Caminho Crítico) - rho =", rho),
    xlab = "Atividade",
    ylab = "Probabilidade (%)",
    ylim = c(0, 100)
  )
  grid(nx = NA, ny = NULL)

  # --- Dispersão prazo x custo ---
  cat("\n--- Correlação Prazo x Custo ---\n")
  plot(res$durCC, res$cusCC,
       main = paste("Dispersão Prazo x Custo (rho =", rho, ")"),
       xlab = "Duração (dias)", ylab = "Custo (R$)",
       pch = 19, col = rgb(0, 0, 1, 0.3))

  reg <- lm(res$cusCC ~ res$durCC)
  abline(reg, col = "red", lwd = 2)

  cat("Correlação amostral (prazo, custo):", cor(res$durCC, res$cusCC), "\n")
  cat("Coeficientes da regressão (custo ~ prazo):\n")
  print(coef(reg))
}


## Comparação entre os valores de ρ

In [ ]:
# Tabela resumo: risco de prazo e custo por rho
resumo <- data.frame(
  rho            = rhos,
  prazo_media    = sapply(resultados, function(r) mean(r$durCC)),
  prazo_dp       = sapply(resultados, function(r) sd(r$durCC)),
  prazo_p85      = sapply(resultados, function(r) quantile(r$durCC, 0.85)),
  custo_media    = sapply(resultados, function(r) mean(r$cusCC)),
  custo_dp       = sapply(resultados, function(r) sd(r$cusCC)),
  custo_p85      = sapply(resultados, function(r) quantile(r$cusCC, 0.85)),
  cor_prazo_custo= sapply(resultados, function(r) cor(r$durCC, r$cusCC))
)

print(resumo)

# Probabilidade de cada atividade estar no caminho crítico, por rho
tabelaProbCC <- sapply(resultados, function(r) r$probCC)
rownames(tabelaProbCC) <- LETTERS[1:10]
print(round(tabelaProbCC, 4))


## Comentário sobre o impacto do coeficiente de correlação no risco

*(Preencher após observar os resultados da tabela `resumo` e dos gráficos gerados para cada ρ.)*

Pontos a discutir:

- **Risco de prazo e de custo (desvio-padrão / amplitude da distribuição):** como `prazo_dp` e `custo_dp` variam com ρ? Espera-se que aumentem com ρ, já que a correlação positiva entre as atividades reduz o efeito de "cancelamento" entre variações independentes (a soma de variáveis correlacionadas tem variância maior que a soma de variáveis independentes, dado pelo termo de covariância).
- **Percentil P85 (prazo e custo):** como o valor de planejamento conservador (P85) se desloca com ρ?
- **Probabilidade de caminho crítico por atividade:** essas probabilidades mudam significativamente entre ρ=0 e ρ=0.95? O que isso indica sobre a robustez da identificação do "caminho crítico" quando há forte correlação entre as atividades?
- **Correlação prazo x custo (`cor_prazo_custo`):** como essa correlação evolui com ρ? Faz sentido que, ao aumentar a correlação entre as atividades, a correlação entre a duração total e o custo total também aumente, pois ambas dependem fortemente das mesmas variáveis subjacentes (o mesmo `U`)?
- **Implicações práticas:** o que esses resultados sugerem para o planejador, ao decidir reservas de contingência de prazo e de orçamento, se as atividades de uma obra real costumam ser positivamente correlacionadas (mesma equipe, mesmo fornecedor, condições climáticas compartilhadas, etc.)?
